# Data Exploration & Audit Notebook

This notebook loads the cleaned `.parquet` data files for 2023 and 2019. 

It performs three main tasks:
1.  **Audit:** Compares the row counts of the *raw* data to the *cleaned* data to see how many rows were filtered out.
2.  **Explore Answers:** Uses the `SVDataFrame` class to inspect the schema, columns, and individual cell values for *voter/candidate answers*.
3.  **Explore Questions:** Loads and inspects the *question text* dataframes.

## 1. Setup Project Path

This cell finds the project's root folder and adds it to the system path. This is necessary so that the next cell can successfully import `from configs.base_constants import ...`

In [2]:
import sys
import os
from pathlib import Path

# Start from the current working directory
cwd = Path.cwd()
PROJECT_ROOT = None

# Search upwards (max 5 levels) for the project root
for _ in range(5):
    # We know we're at the root if we see these folders
    if (cwd / "dependencies").is_dir() and (cwd / "vqs").is_dir():
        PROJECT_ROOT = cwd
        break
    # If not found, go one level up
    cwd = cwd.parent

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root. Please run Jupyter from within your project folder.")

# 1. Add project root to sys.path to find 'dependencies' and 'configs'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    print(f"Added to sys.path: {PROJECT_ROOT}")

# 2. Change CWD to root for easy file access ('data/cleaned')
os.chdir(PROJECT_ROOT)
print(f"Current Working Directory: {os.getcwd()}")

Added to sys.path: c:\Users\liv\git\thesis\vaa-question-similarity
Current Working Directory: c:\Users\liv\git\thesis\vaa-question-similarity


## 2. Load Dependencies & Helper Functions

Now we can import everything from our modules, including our `base_constants`.

**Note:** You must **Restart the Kernel** (circular arrow button ⟳ at the top) if you get an `ImportError` after changing `base_constants.py`.

In [3]:
import pandas as pd
from dependencies import SVDataFrame  # This import should now work!
from pathlib import Path
from typing import Optional

# --- NO MORE DUPLICATION! ---
# We can now import *everything* from our single source of truth.
from configs.base_constants import (
    CLEANED_DIR, 
    RAW_CAND_2023_PATH,
    RAW_VOTERS_2023_PATH, 
    RAW_CAND_2019_PATH,
    RAW_VOTERS_2019_PATH,
    VOTERS_19_PREFIX,
    VOTERS_PREFIX,
    CANDIDATES_19_PREFIX,
    CANDIDATES_PREFIX,
    QUESTIONS_2023_PATH, 
    QUESTIONS_2019_PATH   
)

def load_parquet_by_prefix(directory: Path, prefix: str) -> Optional[SVDataFrame]:
    """
    Finds, loads, and wraps a parquet file in an SVDataFrame.
    """
    print(f"\nSearching for file with prefix: '{prefix}'...")
    files = list(directory.glob(f"{prefix}*.parquet"))
    
    if not files:
        print(f"❌ No file found starting with '{prefix}' in {directory}")
        return None

    file_path = files[0]
    print(f"✅ Found: {file_path.name}")
    try:
        df = pd.read_parquet(file_path)
        sv_df = SVDataFrame(df)
        print(f"Successfully loaded and wrapped in SVDataFrame.")
        return sv_df
    except Exception as e:
        print(f"❌ Error reading or wrapping file: {e}")
        return None

Loaded 'base_constants' in notebook mode.
Local environment detected. Using data path: c:\Users\liv\git\thesis\vaa-question-similarity\data


---

# Section A: 2023 Data

---

## 3.1. Audit 2023 Candidate Data

In [4]:
print("--- AUDITING 2023 CANDIDATE DATA ---")

# 1. Load Raw 2023 Candidates (Before)
try:
    df_cand_raw_2023 = pd.read_csv(RAW_CAND_2023_PATH, low_memory=False)
    raw_count = len(df_cand_raw_2023)
    print(f"'BEFORE' (Raw CSV) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading raw file {RAW_CAND_2023_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2023 Candidates (After)
df_cand = load_parquet_by_prefix(CLEANED_DIR, CANDIDATES_PREFIX)
if df_cand is not None:
    cleaned_count = len(df_cand)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\nFiltered out: {lost_count} candidates ({lost_percent:.2f}%)")

--- AUDITING 2023 CANDIDATE DATA ---
'BEFORE' (Raw CSV) count: 5925 rows

Searching for file with prefix: 'df_candidates-'...
✅ Found: df_candidates-2a875b6e20585f450e0dbdf685e69ece7c7c7f3a245a302053cd1d024a520461.parquet
Successfully loaded and wrapped in SVDataFrame.
'AFTER' (Cleaned Parquet) count: 4983 rows

Filtered out: 942 candidates (15.90%)


## 3.2. Audit 2023 Voter Data

In [5]:
print("--- AUDITING 2023 VOTER DATA ---")

# 1. Load Preprocessed 2023 Voters (Before)
try:
    df_voters_raw_2023 = pd.read_parquet(RAW_VOTERS_2023_PATH)
    raw_count = len(df_voters_raw_2023)
    print(f"'BEFORE' (Preprocessed Parquet) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading preprocessed file {RAW_VOTERS_2023_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2023 Voters (After)
df_voters = load_parquet_by_prefix(CLEANED_DIR, VOTERS_PREFIX)
if df_voters is not None:
    cleaned_count = len(df_voters)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\n--- 2023 VOTER DATA LOSS REPORT ---")
        print(f"Total rows filtered by clean_voters(): {lost_count}")
        print(f"Percentage of 'junk' data removed: {lost_percent:.2f}%")

--- AUDITING 2023 VOTER DATA ---
'BEFORE' (Preprocessed Parquet) count: 299046 rows

Searching for file with prefix: 'df_voters-'...
✅ Found: df_voters-fe0fb9bd69ac41cac556d0195984755037c8021f6cb88df9b5a5d368b1d96bfd.parquet
Successfully loaded and wrapped in SVDataFrame.
'AFTER' (Cleaned Parquet) count: 188162 rows

--- 2023 VOTER DATA LOSS REPORT ---
Total rows filtered by clean_voters(): 110884
Percentage of 'junk' data removed: 37.08%


In [6]:
if df_voters_raw_2023 is not None:
    print(df_voters_raw_2023.loc[800, 'recID'])

f3a77cf7-9a40-44fc-889a-81e7dfa7eb11


## 3.3. Inspect 2023 Data Columns

### Candidates

In [7]:
if df_cand is not None:
    all_columns_cand = df_cand.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_cand)}")
    print("---")
    print(all_columns_cand)

Total columns found: 233
---
['ID_user', 'ID_candidate', 'ID_election', 'ID_district', 'canton', 'ID_election_party', 'party_short', 'ID_district_party', 'party_REC6', 'ID_list', 'list_place_1', 'list_place_2', 'candidate_no_1', 'candidate_no_2', 'cumulated', 'incumbent', 'elected', 'MP', 'firstname', 'lastname', 'full_name', 'zip', 'city', 'country', 'language', 'gender', 'year_of_birth', 'age', 'age_REC6', 'denomination', 'marital_status', 'N_children', 'highest_education', 'occupation', 'employers', 'image_URL', 'video_URL', 'facebook_URL', 'twitter_URL', 'instagram_URL', 'website_URL', 'additional_URL', 'smartspider_png_URL', 'smartspider_pdf_URL', 'smartspider_svg_URL', 'portrait_URL', 'funding_amount', 'funding_comment', 'slogan', 'hobbies', 'fav_books', 'fav_movies', 'fav_music', 'status', 'N_answers', 'answer_32214', 'answer_32215', 'answer_32216', 'answer_32217', 'answer_32218', 'answer_32219', 'answer_32220', 'answer_32221', 'answer_32222', 'answer_32223', 'answer_32224', 'an

In [8]:
# Inspect a single cell value of candidates DataFrame
if df_cand is not None:
    print(df_cand.loc[0, 'ID_district'])

946


### Voters

In [9]:
if df_voters is not None:
    all_columns_voters = df_voters.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_voters)}")
    print("---")
    print(all_columns_voters)

Total columns found: 194
---
['recID', 'user_language', 'voterID', 'researchID', 'electionID', 'districtID', 'sourceTYPE', 'source', 'recTYPE', 'questTYPE', 'language', 'birthYEAR', 'gender', 'zip', 'education', 'interest', 'position', 'pref_party', 'soc_completion', 'N_answers', 'quest_completion', 'answer_32214', 'answer_32215', 'answer_32216', 'answer_32217', 'answer_32218', 'answer_32219', 'answer_32220', 'answer_32221', 'answer_32222', 'answer_32223', 'answer_32224', 'answer_32225', 'answer_32226', 'answer_32227', 'answer_32228', 'answer_32229', 'answer_32230', 'answer_32231', 'answer_32232', 'answer_32233', 'answer_32234', 'answer_32235', 'answer_32236', 'answer_32237', 'answer_32238', 'answer_32239', 'answer_32240', 'answer_32241', 'answer_32242', 'answer_32243', 'answer_32244', 'answer_32245', 'answer_32246', 'answer_32247', 'answer_32248', 'answer_32249', 'answer_32250', 'answer_32251', 'answer_32252', 'answer_32253', 'answer_32254', 'answer_32255', 'answer_32256', 'answer_322

In [10]:
# Inspect row values
if df_voters is not None:
    print(df_voters.loc[3, 'districtID'])

933.0


## 3.4. Inspect 2023 Answer Matrices

In [11]:
if df_cand is not None:
    print("--- 2023 Candidate Answer Matrix (Head) ---")
    display(df_cand.answers().head())

--- 2023 Candidate Answer Matrix (Head) ---


,answer_32214,answer_32215,answer_32216,answer_32217,answer_32218,answer_32219,answer_32220,answer_32221,answer_32222,answer_32223,...,answer_32279,answer_32280,answer_32281,answer_32282,answer_32283,answer_32284,answer_32285,answer_32286,answer_32287,answer_32288
0,0.0,75.0,25.0,0.0,100.0,100.0,25.0,100.0,0.0,25.0,...,33.0,100.0,75.0,75.0,75.0,100.0,50.0,50.0,0.0,75.0
1,75.0,25.0,75.0,25.0,25.0,75.0,25.0,75.0,75.0,100.0,...,67.0,50.0,50.0,50.0,50.0,75.0,50.0,50.0,75.0,75.0
2,0.0,100.0,100.0,0.0,25.0,0.0,100.0,0.0,0.0,0.0,...,50.0,50.0,75.0,75.0,50.0,75.0,100.0,75.0,100.0,0.0
3,0.0,100.0,100.0,0.0,100.0,75.0,75.0,100.0,75.0,25.0,...,33.0,100.0,75.0,100.0,75.0,100.0,0.0,25.0,25.0,75.0
4,0.0,100.0,100.0,0.0,100.0,100.0,0.0,25.0,0.0,75.0,...,0.0,100.0,75.0,75.0,50.0,75.0,25.0,25.0,0.0,75.0


In [12]:
if df_voters is not None:
    print("--- 2023 Voter Answer Matrix (Head) ---")
    display(df_voters.answers().head())

--- 2023 Voter Answer Matrix (Head) ---


,answer_32214,answer_32215,answer_32216,answer_32217,answer_32218,answer_32219,answer_32220,answer_32221,answer_32222,answer_32223,...,answer_32279,answer_32280,answer_32281,answer_32282,answer_32283,answer_32284,answer_32285,answer_32286,answer_32287,answer_32288
0,0.0,75.0,75.0,0.0,75.0,75.0,0.0,75.0,25.0,100.0,...,50.0,83.0,75.0,50.0,50.0,75.0,0.0,75.0,75.0,25.0
1,25.0,75.0,25.0,25.0,100.0,75.0,25.0,100.0,0.0,0.0,...,17.0,67.0,50.0,100.0,25.0,75.0,25.0,50.0,75.0,50.0
2,25.0,100.0,100.0,25.0,100.0,100.0,75.0,0.0,25.0,25.0,...,33.0,100.0,100.0,75.0,75.0,75.0,25.0,75.0,0.0,75.0
3,100.0,75.0,100.0,25.0,75.0,75.0,100.0,0.0,25.0,100.0,...,67.0,33.0,50.0,100.0,75.0,75.0,75.0,100.0,100.0,25.0
4,75.0,75.0,0.0,0.0,75.0,100.0,75.0,100.0,0.0,0.0,...,17.0,83.0,75.0,75.0,25.0,75.0,25.0,25.0,0.0,75.0


In [13]:
# Inspect a single cell value
if 'df_voters' in locals() and df_voters is not None:
    print(df_voters.loc[15, 'recID'])

850340d0-8ffb-43b4-b114-9fa42a74ad60


---

# Section B: 2019 Data

---

## 4.1. Audit 2019 Candidate Data

In [14]:
print("--- AUDITING 2019 CANDIDATE DATA ---")

# 1. Load Raw 2019 Candidates (Before)
try:
    df_cand_raw_2019 = pd.read_csv(RAW_CAND_2019_PATH, low_memory=False)
    raw_count = len(df_cand_raw_2019)
    print(f"'BEFORE' (Raw CSV) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading raw file {RAW_CAND_2019_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2019 Candidates (After)
df_cand19 = load_parquet_by_prefix(CLEANED_DIR, CANDIDATES_19_PREFIX)
if df_cand19 is not None:
    cleaned_count = len(df_cand19)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\nFiltered out: {lost_count} candidates ({lost_percent:.2f}%)")

--- AUDITING 2019 CANDIDATE DATA ---
'BEFORE' (Raw CSV) count: 4663 rows

Searching for file with prefix: 'df_candidates19-'...
✅ Found: df_candidates19-bb7666c18181c2979379e2af52f4c60284b7d6e15609612ea184370daf1888b1.parquet
Successfully loaded and wrapped in SVDataFrame.
'AFTER' (Cleaned Parquet) count: 3926 rows

Filtered out: 737 candidates (15.81%)


## 4.2. Audit 2019 Voter Data

In [15]:
print("--- AUDITING 2019 VOTER DATA ---")

# 1. Load Raw 2019 Voters (Before)
try:
    df_voters_raw_2019 = pd.read_csv(RAW_VOTERS_2019_PATH, low_memory=False)
    raw_count = len(df_voters_raw_2019)
    print(f"'BEFORE' (Raw CSV) count: {raw_count} rows")
except Exception as e:
    print(f"❌ Error loading raw file {RAW_VOTERS_2019_PATH}: {e}")
    raw_count = 0

# 2. Load Cleaned 2019 Voters (After)
df_voters19 = load_parquet_by_prefix(CLEANED_DIR, VOTERS_19_PREFIX)
if df_voters19 is not None:
    cleaned_count = len(df_voters19)
    print(f"'AFTER' (Cleaned Parquet) count: {cleaned_count} rows")

    # 3. Report
    if raw_count > 0:
        lost_count = raw_count - cleaned_count
        lost_percent = (lost_count / raw_count) * 100
        print(f"\n--- 2019 VOTER DATA LOSS REPORT ---")
        print(f"Total rows filtered by clean_voters19(): {lost_count}")
        print(f"Percentage of 'junk' data removed: {lost_percent:.2f}%")

--- AUDITING 2019 VOTER DATA ---
'BEFORE' (Raw CSV) count: 427572 rows

Searching for file with prefix: 'df_voters19-'...
✅ Found: df_voters19-8bf0f790a805189c550b8e463155b00af5aaa0cd618d0eada3115689483c1a6a.parquet
Successfully loaded and wrapped in SVDataFrame.
'AFTER' (Cleaned Parquet) count: 389881 rows

--- 2019 VOTER DATA LOSS REPORT ---
Total rows filtered by clean_voters19(): 37691
Percentage of 'junk' data removed: 8.82%


## 4.3. Inspect 2019 Candidate Columns

In [16]:
if 'df_cand19' in locals() and df_cand19 is not None:
    all_columns_cand19 = df_cand19.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_cand19)}")
    print("---")
    print(all_columns_cand19)

Total columns found: 188
---
['ID_election', 'ID_candidate', 'gender', 'year_of_birth', 'zip', 'city', 'country', 'language', 'candidate_NR&SR', 'ID_party', 'party_short', 'ID_district', 'district', 'ID_list', 'list', 'list_place_1', 'candidate_no_1', 'list_place_2', 'incumbent_NR', 'elected_NR', 'occupation', 'confirmed', 'n_answers', 'answer_3387', 'answer_3388', 'answer_3389', 'answer_3391', 'answer_3392', 'answer_3398', 'answer_3399', 'answer_3412', 'answer_3413', 'answer_3414', 'answer_3415', 'answer_3416', 'answer_3417', 'answer_3418', 'answer_3419', 'answer_3420', 'answer_3421', 'answer_3422', 'answer_3423', 'answer_3424', 'answer_3425', 'answer_3426', 'answer_3427', 'answer_3428', 'answer_3429', 'answer_3430', 'answer_3431', 'answer_3432', 'answer_3433', 'answer_3434', 'answer_3435', 'answer_3436', 'answer_3437', 'answer_3438', 'answer_3439', 'answer_3440', 'answer_3441', 'answer_3442', 'answer_3443', 'answer_3444', 'answer_3445', 'answer_3446', 'answer_3447', 'answer_3448', 'a

In [17]:
# Inspect a single cell value
if 'df_cand19' in locals() and df_cand19 is not None:
    try:
        # Get the first answer column name automatically
        print(f"\n--- Inspecting single cell --- ")
        print(f"Random cell value at [Row 5, Column 'ID_district']: ", end="")
        print(df_cand19.loc[5, 'ID_district'])
    except Exception as e:
        print(f"Error inspecting cell: {e}")


--- Inspecting single cell --- 
Random cell value at [Row 5, Column 'ID_district']: 25.0


## 4.4 Inspecting 2019 Voter Columns

In [18]:
if 'df_voters19' in locals() and df_voters19 is not None:
    all_columns_voters19 = df_voters19.columns.tolist()
    
    print(f"Total columns found: {len(all_columns_voters19)}")
    print("---")
    print(all_columns_voters19)

Total columns found: 189
---
['ID_recommendation', 'ID_user', 'ID_voter', 'ID_election', 'ID_district', 'timestamp', 'timestamp_REC', 'source', 'questionnaire_type', 'recommendation_type', 'language', 'gender', 'year_of_birth', 'year_of_birth_REC', 'education', 'zip', 'pol_interest', 'pol_position', 'party', 'n_answers', 'answer_3412', 'answer_3413', 'answer_3414', 'answer_3415', 'answer_3416', 'answer_3417', 'answer_3418', 'answer_3419', 'answer_3420', 'answer_3421', 'answer_3422', 'answer_3423', 'answer_3424', 'answer_3425', 'answer_3426', 'answer_3427', 'answer_3428', 'answer_3429', 'answer_3430', 'answer_3431', 'answer_3391', 'answer_3392', 'answer_3432', 'answer_3433', 'answer_3434', 'answer_3435', 'answer_3436', 'answer_3437', 'answer_3438', 'answer_3439', 'answer_3440', 'answer_3441', 'answer_3442', 'answer_3443', 'answer_3444', 'answer_3445', 'answer_3446', 'answer_3447', 'answer_3448', 'answer_3449', 'answer_3450', 'answer_3451', 'answer_3452', 'answer_3453', 'answer_3454', 'a

In [19]:
# Inspect a single cell value
if 'df_voters19' in locals() and df_voters19 is not None:
    try:
        # Get the first answer column name automatically
        print(f"\n--- Inspecting single cell --- ")
        print(f"Value at [Row 0, Column 'ID_district']: ", end="")
        print(df_voters19.loc[0, 'ID_district'])
    except Exception as e:
        print(f"Error inspecting cell: {e}")


--- Inspecting single cell --- 
Value at [Row 0, Column 'ID_district']: 8


## 4.4. Inspect 2019 Answer Matrices

In [20]:
if 'df_cand19' in locals() and df_cand19 is not None:
    print("--- 2019 Candidate Answer Matrix (Head) ---")
    display(df_cand19.answers().head())

--- 2019 Candidate Answer Matrix (Head) ---


,answer_3387,answer_3388,answer_3389,answer_3391,answer_3392,answer_3398,answer_3399,answer_3412,answer_3413,answer_3414,...,answer_3470,answer_3471,answer_3472,answer_3473,answer_3474,answer_3475,answer_3476,answer_3477,answer_3478,answer_3479
1,67.0,33.0,33.0,0.0,25.0,0.0,83.0,0.0,25.0,100.0,...,100.0,100.0,75.0,50.0,75.0,75.0,50.0,50.0,75.0,75.0
2,67.0,33.0,33.0,0.0,0.0,0.0,67.0,25.0,100.0,75.0,...,100.0,100.0,75.0,25.0,50.0,50.0,50.0,50.0,75.0,50.0
3,67.0,33.0,50.0,0.0,0.0,0.0,67.0,25.0,100.0,100.0,...,100.0,100.0,75.0,50.0,50.0,75.0,50.0,50.0,75.0,50.0
4,67.0,33.0,33.0,0.0,25.0,0.0,67.0,75.0,100.0,100.0,...,100.0,100.0,50.0,50.0,50.0,75.0,50.0,50.0,75.0,50.0
5,83.0,33.0,0.0,100.0,0.0,0.0,67.0,0.0,100.0,100.0,...,100.0,100.0,75.0,50.0,50.0,50.0,50.0,50.0,75.0,50.0


In [21]:
if 'df_voters19' in locals() and df_voters19 is not None:
    print("--- 2019 Voter Answer Matrix (Head) ---")
    display(df_voters19.answers().head())

--- 2019 Voter Answer Matrix (Head) ---


,answer_3387,answer_3388,answer_3389,answer_3391,answer_3392,answer_3398,answer_3399,answer_3412,answer_3413,answer_3414,...,answer_3470,answer_3471,answer_3472,answer_3473,answer_3474,answer_3475,answer_3476,answer_3477,answer_3478,answer_3479
0,17.0,50.0,33.0,25.0,75.0,0.0,83.0,100.0,75.0,75.0,...,25.0,0.0,25.0,50.0,50.0,50.0,75.0,50.0,50.0,50.0
1,83.0,83.0,83.0,75.0,0.0,0.0,83.0,25.0,75.0,100.0,...,100.0,100.0,75.0,0.0,50.0,100.0,50.0,50.0,50.0,25.0
2,0.0,0.0,100.0,100.0,100.0,0.0,0.0,0.0,100.0,100.0,...,100.0,100.0,75.0,100.0,25.0,50.0,100.0,100.0,100.0,75.0
3,0.0,0.0,0.0,100.0,0.0,0.0,100.0,0.0,100.0,100.0,...,100.0,100.0,100.0,0.0,50.0,100.0,100.0,50.0,100.0,50.0
4,100.0,100.0,83.0,25.0,0.0,75.0,67.0,100.0,0.0,0.0,...,100.0,75.0,75.0,25.0,75.0,75.0,25.0,25.0,75.0,100.0


---

# Section C: Question Text Data

This section loads the question text dataframes we created with the `build_question_dfs.py` script. This text is the input for SBERT.

## 5.1. Load & Inspect 2023 Questions

In [22]:
print("--- LOADING 2023 QUESTIONS ---")

try:
    df_questions = pd.read_parquet(QUESTIONS_2023_PATH)
    print(f"Shape: {df_questions.shape}")
except Exception as e:
    print(f"❌ Error loading {QUESTIONS_2023_PATH}: {e}")

--- LOADING 2023 QUESTIONS ---
Shape: (75, 27)


In [23]:
# Explore the 2023 questions
print("--- 2023 Question Columns ---")
print(df_questions.columns.tolist())

--- 2023 Question Columns ---
['ID_election', 'ID_question', 'question_DE', 'question_FR', 'question_IT', 'question_EN', 'category', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'type', 'rapide', 'cleavage_1', 'cleavage_2', 'cleavage_3', 'cleavage_4', 'cleavage_5', 'cleavage_6', 'cleavage_7', 'cleavage_8', 'info_DE', 'pro_DE', 'contra_DE', '_category', '_n_options']


In [24]:
print("\n--- 2023 Question Data (Head) ---")
display(df_questions.head())


--- 2023 Question Data (Head) ---


,ID_election,ID_question,question_DE,question_FR,question_IT,question_EN,category,tag_1,tag_2,tag_3,...,cleavage_4,cleavage_5,cleavage_6,cleavage_7,cleavage_8,info_DE,pro_DE,contra_DE,_category,_n_options
0,1057,32214,Befürworten Sie eine Erhöhung des Rentenalters...,Êtes-vous favorable à une augmentation de l'âg...,È favorevole all’innalzamento dell’età pension...,Do you support an increase in the retirement a...,11451,2,1,168.0,...,NaN,NaN,NaN,-1.0,NaN,<p>Die Alters- und Hinterlassenenversicherung ...,"<p dir=""ltr"">Das Rentensystem muss aufgrund de...","<p dir=""ltr"">Die Erh&ouml;hung des Rentenalter...",Welfare state & family,4
1,1057,32215,Soll der Staat mehr Mittel für die Krankenkass...,L'État doit-il allouer davantage de moyens à l...,Ritiene che lo Stato dovrebbe stanziare più me...,Should the federal government allocate more fu...,11451,168,122,7.0,...,NaN,NaN,NaN,1.0,NaN,"<p dir=""ltr"">Die Krankenkassenpr&auml;mie ist ...","<p dir=""ltr"">Die Kosten der Krankenkassenpr&au...","<p dir=""ltr"">Die Kantone sind zust&auml;ndig f...",Welfare state & family,4
2,1057,32216,Bei Ehepaaren ist die Höhe der Rente heute auf...,"Pour les couples mariés, le montant de la rent...","Per le coppie di coniugi, l’ammontare della re...","For married couples, the pension is currently ...",11451,168,169,171.0,...,NaN,NaN,NaN,1.0,NaN,"<p dir=""ltr"">Die AHV ist die Erste S&auml;ule ...","<p dir=""ltr"">Die Plafonierung der AHV benachte...","<p dir=""ltr"">Die Aufhebung der Plafonierung be...",Welfare state & family,4
3,1057,32217,Im Rahmen der BVG-Reform sollen die Renten gek...,"Dans le cadre de la réforme de la LPP, les ren...",La riforma della LPP prevede una riduzione del...,As part of the reform of the BVG (occupational...,11451,168,170,171.0,...,NaN,NaN,NaN,-1.0,NaN,"<p dir=""ltr"">Die obligatorische berufliche Vor...",None,None,Welfare state & family,4
4,1057,32218,Soll die bezahlte Elternzeit von heute 14 Woch...,Faut-il étendre le congé rémunéré pour les par...,Ritiene che il congedo parentale retribuito do...,Should paid parental leave be increased beyond...,11451,168,3,172.0,...,NaN,NaN,NaN,1.0,NaN,"<p dir=""ltr"">In der Schweiz k&ouml;nnen Frauen...","<ul>\n<li dir=""ltr"" aria-level=""1"">\n<p dir=""l...","<ul>\n<li dir=""ltr"" aria-level=""1"">\n<p dir=""l...",Welfare state & family,4


In [25]:
# print a sample question
qu = df_questions.get('question_EN', ['No "question_EN" column found.'])[0]
print(qu)

Do you support an increase in the retirement age (e.g., to 67)?


In [31]:
print(df_questions["_n_options"].unique())

[4 7 5]


## 5.2. Load & Inspect 2019 Questions

Note: Rows start at index 1 for some reason.

In [26]:
print("--- LOADING 2019 QUESTIONS ---")

try:
    df_questions19 = pd.read_parquet(QUESTIONS_2019_PATH)
    print(f"Shape: {df_questions19.shape}")
except Exception as e:
    print(f"❌ Error loading {QUESTIONS_2019_PATH}: {e}")

--- LOADING 2019 QUESTIONS ---
Shape: (75, 6)


In [27]:
# Explore the 2019 questions
print("--- 2019 Question Columns ---")
print(df_questions19.columns.tolist())

--- 2019 Question Columns ---
['ID_question', 'type', 'question_de', 'question_fr', 'question_en', '_n_options']


In [28]:
print("\n--- 2019 Question Data (Head) ---")
display(df_questions19.head())


--- 2019 Question Data (Head) ---


,ID_question,type,question_de,question_fr,question_en,_n_options
1,3387,Slider-7,"Wie beurteilen Sie diese Aussage: ""Wer sich ni...","Comment jugez-vous l'affirmation suivante: ""Qu...",What is your position the following statement:...,7
2,3388,Slider-7,"Wie beurteilen Sie diese Aussage: ""Die Bestraf...","Comment jugez-vous l'affirmation suivante: ""Pu...",What is your position the following statement:...,7
3,3389,Slider-7,"Wie beurteilen Sie diese Aussage: ""Für ein Kin...","Comment jugez-vous l'affirmation suivante: ""Po...",What is your position the following statement:...,7
4,3391,Standard-4,Soll der Bund Ausländer/-innen bei der Integra...,La Confédération devrait-elle soutenir davanta...,Should the federal government provide more sup...,4
5,3392,Standard-4,Soll der Konsum von Cannabis legalisiert werden?,La consommation de cannabis devrait-elle être ...,Should cannabis use be legalized?,4


In [29]:
print(df_questions19["type"].unique())

['Slider-7' 'Standard-4' 'Budget-5']


In [25]:
print("\n--- Example Question (for SBERT) ---")
first_q_text_2019 = df_questions19.get('question_en')[1] # row starts from 1 for some reason
print(first_q_text_2019)


--- Example Question (for SBERT) ---
What is your position the following statement: "Someone who is not guilty, has nothing to fear from state security measures."."
